<a href="https://colab.research.google.com/github/igorfantucci/Aula-Automatica---GRUPO-5/blob/main/etapa-03-arvores/Aula%2019%20-%20Algoritmo%20de%20Kruskal%20e%20Prim%20-%20code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Otimização de Infraestrutura na Planta de Biodiesel: Árvores Geradoras Mínimas (MST)
## Dimensionamento de Malhas de Utilidades (Pipe Racks) e Barramento SCADA via Algoritmos de Kruskal e Prim

---

### 1. Contexto Industrial e Engenharia de Processos (Grupo 5)
Na **Planta Piloto de Produção de Biodiesel por Transesterificação em Batelada**, a integração física entre os quatro setores da planta exige um leito central de tubulações (*pipe rack*) e bandejamentos de cabeamento elétrico e de rede SCADA.

Os quatro setores fabris modelados são:
- **Setor 100 (Armazenamento e Preparação):** Silo de Óleo (`TK-101`), Tanque de Metanol (`TK-102`) e Vaso de Preparo do Metóxido (`TK-103`);
- **Setor 200 (Reação Crítica CSTR):** Reator Encamisado (`R-200`) e Tanque Pulmão/Arrefecimento (`TK-201`);
- **Setor 300 (Decantação e Separação):** Decantador Cônico Bifásico (`TK-300`);
- **Setor 400 (Purificação e Armazenamento Final):** Coluna de Lavagem (`COL-401`) e Tanque de Biodiesel Puro B100 (`TK-402`).

Construir conexões físicas redundantes ponto a ponto entre todas as unidades dispararia os custos de montagem mecânica (aço estrutural, bandejas e isolamento térmico), além de onerar a transmissão de sinais de automação. Aplicamos os algoritmos de **Kruskal** e **Prim** para encontrar a **Árvore Geradora Mínima (MST)**, interligando todos os 8 nós com a menor metragem linear contínua possível.

---
### 2. Fundamentação dos Algoritmos de Árvore Geradora Mínima

#### A. Algoritmo de Kruskal (Abordagem Centrada em Arestas)
- **Estratégia Gulosa (*Greedy*):** Ordena todas as arestas do grafo em ordem crescente de comprimento métrico $w(e)$.
- **Inclusão Condicional:** Adiciona iterativamente cada aresta candidata à solução se e somente se suas extremidades pertencerem a componentes separadas, prevenindo loops fechados.
- **Estrutura de Dados:** Utiliza **Union-Find (*Disjoint-Set Union*)** com otimizações de compressão de caminho e união por rank, atingindo tempo quase linear $\mathcal{O}(|E| \log |E|)$.

#### B. Algoritmo de Prim (Abordagem Centrada em Vértices)
- **Crescimento por Arborescência:** Parte de um nó raiz inicial (ex.: `TK-101`) e expande uma única árvore conexa.
- **Fronteira de Corte:** A cada passo, avalia todas as arestas que cruzam o corte $(S, V \setminus S)$ e seleciona a aresta de menor peso incidente a um nó ainda não visitado.
- **Estrutura de Dados:** Utiliza uma fila de prioridade binária (**Min-Heap** via `heapq`), operando em $\mathcal{O}((|E| + |V|) \log |V|)$.

In [1]:
# Importação de Bibliotecas Standard e Terceiros
from typing import List, Tuple, Dict, Set
import heapq
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

# Configuração de estilo dos gráficos
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Bibliotecas de automacao e grafos carregadas com sucesso.")

Bibliotecas de automacao e grafos carregadas com sucesso.


---
### 3. Implementação Orientada a Objetos: Estruturas de Dados e Algoritmos

Definimos abaixo a classe `DisjointSet` e as funções funcionais puras `kruskal_mst` e `prim_mst` com rastreabilidade completa de logs para auditoria de engenharia.

In [2]:
class DisjointSet:
    """
    Estrutura de Dados Union-Find com Otimizacoes de:
    - Compressao de Caminho (Path Compression)
    - Uniao por Rank (Union by Rank)
    """
    def __init__(self, vertices: List[str]):
        self.parent: Dict[str, str] = {v: v for v in vertices}
        self.rank: Dict[str, int] = {v: 0 for v in vertices}

    def find(self, item: str) -> str:
        """Encontra a raiz representativa com compressao de caminho."""
        if self.parent[item] != item:
            self.parent[item] = self.find(self.parent[item])
        return self.parent[item]

    def union(self, root1: str, root2: str) -> bool:
        """Une duas componentes. Retorna True se uniu, False se ja no mesmo componente (ciclo)."""
        r1 = self.find(root1)
        r2 = self.find(root2)
        if r1 == r2:
            return False

        if self.rank[r1] < self.rank[r2]:
            self.parent[r1] = r2
        elif self.rank[r1] > self.rank[r2]:
            self.parent[r2] = r1
        else:
            self.parent[r2] = r1
            self.rank[r1] += 1
        return True

def kruskal_mst(vertices: List[str], edges: List[Tuple[str, str, float]]) -> Tuple[List[Tuple[str, str, float]], float, List[Dict]]:
    """Executa o Algoritmo de Kruskal retornando a MST, peso total e log de decisao."""
    sorted_edges = sorted(edges, key=lambda x: x[2])
    dsu = DisjointSet(vertices)
    mst_edges: List[Tuple[str, str, float]] = []
    total_weight: float = 0.0
    execution_log: List[Dict] = []

    for u, v, w in sorted_edges:
        root_u = dsu.find(u)
        root_v = dsu.find(v)
        if root_u != root_v:
            dsu.union(u, v)
            mst_edges.append((u, v, w))
            total_weight += w
            execution_log.append({
                "Passo": len(execution_log) + 1,
                "Aresta": f"{u} - {v}",
                "Distancia (m)": w,
                "Status": "INCLUIDA",
                "Motivo Engenharia": "Conecta setores/componentes independentes"
            })
        else:
            execution_log.append({
                "Passo": len(execution_log) + 1,
                "Aresta": f"{u} - {v}",
                "Distancia (m)": w,
                "Status": "REJEITADA",
                "Motivo Engenharia": "Formaria ciclo fechado redundante"
            })

        if len(mst_edges) == len(vertices) - 1:
            break

    return mst_edges, total_weight, execution_log

def prim_mst(vertices: List[str], edges: List[Tuple[str, str, float]], start_node: str) -> Tuple[List[Tuple[str, str, float]], float, List[Dict]]:
    """Executa o Algoritmo de Prim utilizando Min-Heap binario."""
    adj: Dict[str, List[Tuple[float, str]]] = {v: [] for v in vertices}
    for u, v, w in edges:
        adj[u].append((w, v))
        adj[v].append((w, u))

    visited: Set[str] = {start_node}
    min_heap: List[Tuple[float, str, str]] = []
    for w, neighbor in adj[start_node]:
        heapq.heappush(min_heap, (w, start_node, neighbor))

    mst_edges: List[Tuple[str, str, float]] = []
    total_weight: float = 0.0
    execution_log: List[Dict] = []
    step = 1

    while min_heap and len(visited) < len(vertices):
        w, u, v = heapq.heappop(min_heap)
        if v in visited:
            execution_log.append({
                "Iteracao": step,
                "Aresta Analisada": f"{u} - {v}",
                "Distancia (m)": w,
                "Decisao": "IGNORADA",
                "Vertice Destino": v,
                "Motivo": f"{v} ja integrado a malha de instalacao"
            })
            step += 1
            continue

        visited.add(v)
        mst_edges.append((u, v, w))
        total_weight += w
        execution_log.append({
            "Iteracao": step,
            "Aresta Analisada": f"{u} - {v}",
            "Distancia (m)": w,
            "Decisao": "INCLUIDA",
            "Vertice Destino": v,
            "Motivo": f"Menor aresta de corte conectando {v}"
        })
        step += 1

        for next_w, neighbor in adj[v]:
            if neighbor not in visited:
                heapq.heappush(min_heap, (next_w, v, neighbor))

    return mst_edges, total_weight, execution_log

---
### 4. Topologia e Rotas Candidatas da Planta de Biodiesel (Grupo 5)

Mapeamos as 8 unidades industriais da fábrica e os 14 trechos viáveis de *pipe racks* com suas distâncias reais em metros:

In [3]:
# Lista de Unidades Industriais da Planta de Biodiesel (Vértices)
nodes = [
    "TK-101",   # Tanque de Óleo Vegetal Degomado (Setor 100)
    "TK-102",   # Vaso de Metanol Anidro (Setor 100)
    "TK-103",   # Tanque de Metóxido de Sódio (Setor 100)
    "R-200",    # Reator CSTR Encamisado (Setor 200)
    "TK-201",   # Vaso Pulmão / Arrefecimento (Setor 200)
    "TK-300",   # Decantador Cônico de Fases (Setor 300)
    "COL-401",  # Coluna de Lavagem e Neutralização (Setor 400)
    "TK-402"    # Tanque de Biodiesel Puro B100 (Setor 400)
]

# Rotas de Pipe Rack e Canaletas de Barramento com Distâncias Físicas (m)
edges = [
    ("TK-102", "TK-103", 10.0),
    ("TK-101", "TK-102", 12.0),
    ("R-200", "TK-201", 14.0),
    ("COL-401", "TK-402", 15.0),
    ("TK-101", "TK-103", 16.0),
    ("TK-201", "TK-300", 18.0),
    ("TK-300", "COL-401", 20.0),
    ("TK-103", "R-200", 22.0),
    ("R-200", "TK-300", 26.0),
    ("TK-101", "R-200", 28.0),
    ("TK-102", "R-200", 30.0),
    ("TK-201", "COL-401", 32.0),
    ("TK-300", "TK-402", 35.0),
    ("R-200", "COL-401", 40.0)
]

total_initial_length = sum(w for _, _, w in edges)
print(f"Total de Unidades Industriais (Vertices): {len(nodes)}")
print(f"Total de Rotas Candidatas (Arestas): {len(edges)}")
print(f"Extensao Bruta da Malha Completa: {total_initial_length:.1f} m")

Total de Unidades Industriais (Vertices): 8
Total de Rotas Candidatas (Arestas): 14
Extensao Bruta da Malha Completa: 318.0 m


---
### 5. Execução dos Algoritmos e Verificação de Consistência

Executamos ambos os algoritmos e validamos a equivalência exata dos resultados métricos.

In [4]:
# Executando Kruskal
kruskal_edges, kruskal_weight, kruskal_log = kruskal_mst(nodes, edges)

# Executando Prim com âncora no Silo de Entrada TK-101
prim_edges, prim_weight, prim_log = prim_mst(nodes, edges, start_node="TK-101")

print("=" * 60)
print(" RESULTADOS DA OTIMIZACAO TOPOLOGICA DA PLANTA DE BIODIESEL")
print("=" * 60)
print(f"Extensao Total Original:             {total_initial_length:.2f} metros")
print(f"Extensao Otima Kruskal:              {kruskal_weight:.2f} metros ({len(kruskal_edges)} arestas)")
print(f"Extensao Otima Prim (Raiz TK-101):   {prim_weight:.2f} metros ({len(prim_edges)} arestas)")
capex_savings = 100 * (1 - kruskal_weight / total_initial_length)
print(f"Economia Fisica Linear (CAPEX):      {capex_savings:.2f}% de reducao")

assert kruskal_weight == 111.0, f"Erro: Kruskal deu {kruskal_weight}"
assert prim_weight == 111.0, f"Erro: Prim deu {prim_weight}"
assert len(kruskal_edges) == len(nodes) - 1
assert len(prim_edges) == len(nodes) - 1
print("Validacao Formal de Equivalencia:    SUCESSO! Kruskal == Prim")

 RESULTADOS DA OTIMIZACAO TOPOLOGICA DA PLANTA DE BIODIESEL
Extensao Total Original:             318.00 metros
Extensao Otima Kruskal:              111.00 metros (7 arestas)
Extensao Otima Prim (Raiz TK-101):   111.00 metros (7 arestas)
Economia Fisica Linear (CAPEX):      65.09% de reducao
Validacao Formal de Equivalencia:    SUCESSO! Kruskal == Prim


#### Tabela 1: Histórico de Decisões do Algoritmo de Kruskal (Union-Find)

In [5]:
df_kruskal = pd.DataFrame(kruskal_log)
try:
    display(df_kruskal)
except NameError:
    print(df_kruskal.to_string(index=False))

#### Tabela 2: Histórico de Expansão por Min-Heap do Algoritmo de Prim

In [6]:
df_prim = pd.DataFrame(prim_log)
try:
    display(df_prim)
except NameError:
    print(df_prim.to_string(index=False))

---
### 6. Visualização Gráfica da Topologia da Planta e Árvore Geradora Mínima

Geramos abaixo o diagrama esquemático com coordenadas espaciais correspondentes aos 4 setores da planta de biodiesel:
- **Setor 100:** Amarelo/Dourado (Armazenamento de Matérias-Primas);
- **Setor 200:** Vermelho/Coral (Reator CSTR e Vaso Pulmão);
- **Setor 300:** Azul (Decantador Cônico);
- **Setor 400:** Verde Esmeralda (Purificação e Expedição B100).

As arestas da MST são desenhadas em linhas verdes grossas contínuas, enquanto as rotas descartadas aparecem em linhas tracejadas cinza.

In [7]:
# Criação do Grafo NetworkX
G = nx.Graph()
for u, v, w in edges:
    G.add_edge(u, v, weight=w)

# Layout espacial dos setores da planta de biodiesel
pos = {
    "TK-101": (0.0, 3.0),   # Setor 100
    "TK-102": (0.0, 1.0),   # Setor 100
    "TK-103": (1.5, 2.0),   # Setor 100
    "R-200":  (3.5, 2.5),   # Setor 200 (Reator CSTR)
    "TK-201": (3.5, 0.8),   # Setor 200 (Tanque Pulmão)
    "TK-300": (5.5, 1.5),   # Setor 300 (Decantador)
    "COL-401": (7.5, 2.5),  # Setor 400 (Lavagem)
    "TK-402": (7.5, 0.8)    # Setor 400 (Tanque B100)
}

# Cores dos nós por setor
node_colors = [
    "#f39c12",  # TK-101 (S100)
    "#f39c12",  # TK-102 (S100)
    "#f39c12",  # TK-103 (S100)
    "#e74c3c",  # R-200  (S200)
    "#e74c3c",  # TK-201 (S200)
    "#3498db",  # TK-300 (S300)
    "#2ecc71",  # COL-401(S400)
    "#27ae60"   # TK-402 (S400)
]

# Conjunto de arestas da MST para verificação rápida
mst_edge_set = set()
for u, v, _ in kruskal_edges:
    mst_edge_set.add((u, v))
    mst_edge_set.add((v, u))

included_edges = [(u, v) for u, v, _ in edges if (u, v) in mst_edge_set or (v, u) in mst_edge_set]
discarded_edges = [(u, v) for u, v, _ in edges if (u, v) not in mst_edge_set and (v, u) not in mst_edge_set]

plt.figure(figsize=(13, 7))

# Desenhar arestas rejeitadas (tracejado fino cinza)
nx.draw_networkx_edges(G, pos, edgelist=discarded_edges, style="dashed", edge_color="#bdc3c7", width=1.5, alpha=0.7)

# Desenhar arestas da MST (verde esmeralda robusto)
nx.draw_networkx_edges(G, pos, edgelist=included_edges, edge_color="#27ae60", width=3.5)

# Desenhar nós
nx.draw_networkx_nodes(G, pos, nodelist=nodes, node_color=node_colors, node_size=2800, edgecolors="#2c3e50", linewidths=2)
nx.draw_networkx_labels(G, pos, font_size=10, font_weight="bold", font_color="white")

# Rótulos das distâncias nas arestas
edge_labels = {(u, v): f"{w:.0f}m" for u, v, w in edges}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=9, font_color="#2c3e50")

plt.title("Topologia de Pipe Racks e Barramento SCADA: Arvore Geradora Minima (MST) - Grupo 5", fontsize=13, fontweight="bold")
plt.axis("off")
plt.tight_layout()
plt.show()
print("Grafico gerado com sucesso.")

---
### 7. Conclusões e Recomendações de Engenharia

1. **Economia Expressiva de CAPEX:** A malha inicial redundante demandaria **318 metros** lineares de leitos estruturais e bandejamentos de cabos. A solução MST reduz esse valor para **111 metros**, proporcionando uma **economia direta de 65.09%** em aço, cabos blindados de instrumentação e isolamento térmico.
2. **Mitigação de Ruído e Integridade de Sinal SCADA:** Em barramentos seriais (RS-485 / Modbus) e analógicos de 4-20 mA HART, a diminuição da metragem total reduz as reflexões de linha e a impedância distribuída, garantindo tempos de varredura (*scan cycle*) mais determinísticos.
3. **Aplicabilidade Contratual:**
   - **Kruskal:** Recomendado para o projeto conceitual (*FEED*), elaboração de listas de materiais e estimativas financeiras orçamentárias;
   - **Prim:** Recomendado para a montagem de campo pelas empreiteiras eletromecânicas, partindo da frente de obras do Setor 100 (`TK-101`) e avançando de maneira contínua até a expedição no Setor 400 (`TK-402`).